# Results for model: mistralai/ministral-14b-instruct-2512

In [1]:
import polars as pl
import pandas as pd
import numpy as np
from xgboost import XGBRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import StandardScaler
import gc

def feature_engineering(df: pl.DataFrame) -> pl.DataFrame:
    global_fields = df.select(pl.repeat('feature_00', repeats=10, alias='feature_00').over(df['date_id'], by="time_id"))
    global_fields = global_fields.select(["date_id", "feature_00"])

    responder_top = df.groupby("date_id").agg(pl.partitions pots=7).top_k_in_group(n=15)
    responder_sorted = df.join(responder_top.select(["date_id", "index"]), on=["date_id", "index"], how="left")
    responder_top_values = responder_sorted.drop("index").filter(pl.col("privilege_escalated") > 0).groupby("date_id").agg(pl.col("feature_00").mean())

    global_sort = pl.DataFrame(global_fields).join(
        responder_top_values.drop_nulls().with_columns(pl.lit(" cuartile_section_topresponder").alias("top_quartile")),
        on="date_id", how="left"
    ).select(["date_id", pl.col("feature_00").sort().suffix("_sorted")])

    # Create quartile flags within each date_id
    quartile_segments = global_sort.with_columns(
        pl.when(pl.col(f"feature_00_sorted") <= pl.col(f"feature_00_sorted").quantile(0.25, subset=plCol))
        .then(pl.lit("q1")).when(pl.col(f"feature_00_sorted") <= pl.col(f"feature_00_sorted").quantile(0.5, subset=plCol))
        .then(pl.lit("q2")).when(pl.col(f"feature_00_sorted") <= pl.col(f"feature_00_sorted").quantile(0.75, subset=plCol))
        .otherwise(pl.lit("q3")).alias("top15_target")
    )

    # Merge and add bin flags back to original
    final_df = df.join(quartile_segments.drop_nulls().filter(pl.col("top15_target") == "q1"), on=["date_id", "time_id"], how="left")
    return final_df.with_columns(
        pl.col("top15_target").cast(pl.Boolean).alias("top_15")
    )

def main():
    # Load data
    train_df = pl.read_parquet("./jane_street_data/train.parquet")

    # Feature engineering
    train_df = feature_engineering(train_df)

    # Standard scalar
    scaler = StandardScaler()
    cat_columns = train_df.select(pl.col(dtype=pl.Utf8).to_list()).to_dict().keys()
    feature_cols = [c for c in train_df.columns if c not in cat_columns + ["date_id", "time_id", "responder_6"]]
    numeric_train = train_df[feature_cols].to_numpy()

    scaler.fit(numeric_train)
    scaled_numerics = scaler.transform(numeric_train)
    numeric_df = pl.from_numpy(scaled_numerics, schema=feature_cols)
    train_df = train_df.drop(feature_cols).join(numeric_df)

    # Final trejset and model train
    train_full, validation = train_test_split(train_df, test_size=0.2, random_state=42, stratify=train_df["responder_6"].cast(pl.Int32))

    model = XGBRegressor(n_estimators=100, n_jobs=4,
                         random_state=42, verbose=0,
                         max_depth=8,
                         early_stopping_rounds=20,
                         eval_metric="rmse")
    model.fit(train_full[train_full.columns[:150]],
              train_full["responder_6"],
              early_stopping_rounds=20,
              eval_set=[(validation[validation.columns[:150]], validation["responder_6"])],
              verbose=False)
    del numeric_df
    gc.collect()

if __name__ == "__main__":
    main()

SyntaxError: invalid syntax. Perhaps you forgot a comma? (3837139817.py, line 14)